# VAPAR (UZH) + Spyx Architectures

This notebook targets the VAPAR dataset/repo from UZH.

Status in Tonic:
- No VAPAR dataset class in `tonic.datasets` in this environment.

Goal:
- Load local VAPAR-style frame/trajectory data.
- Evaluate Spyx architectures on trajectory prediction / control surrogates.

In [ ]:
from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

import spyx.models as fm

DATA_ROOT = Path("../data/VAPAR")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT.resolve())


def tonic_has_vapar():
    import tonic.datasets as d
    return any("vapar" in n.lower() for n in dir(d))

print("tonic_has_vapar =", tonic_has_vapar())
print("expected local artifacts from VAPAR/OSF: drone.csv, frame_index.csv, split*.csv")

In [ ]:
def synthetic_vapar_batch(batch=4, T=12, H=90, W=160, C=3, seed=3):
    rng = np.random.default_rng(seed)
    x = rng.random(size=(T, batch, H, W, C), dtype=np.float32)
    y = rng.normal(size=(batch, 3)).astype(np.float32)
    return jnp.asarray(x), jnp.asarray(y)


def vapar_model_bank(input_hw=(90, 160), input_channels=3):
    conv_cfg = fm.ConvConfig(input_hw=input_hw, input_channels=input_channels, channels1=24, channels2=48, output_dim=3)
    logpolar_cfg = fm.LogPolarFoveatedConvConfig(
        input_hw=input_hw,
        input_channels=input_channels,
        radial_bins=24,
        angular_bins=48,
        channels1=24,
        channels2=48,
        output_dim=3,
    )
    return {
        "conv_lif": lambda x: fm.ConvLIFSNN(conv_cfg)(x),
        "logpolar_foveated": lambda x: fm.LogPolarFoveatedConvSNN(logpolar_cfg)(x),
        "ternary_conv": lambda x: fm.TernaryConvLIFSNN(conv_cfg)(x),
    }


x, y = synthetic_vapar_batch()
for name, fn in vapar_model_bank().items():
    net = hk.without_apply_rng(hk.transform(fn))
    params = net.init(jax.random.PRNGKey(0), x)
    pred, aux = net.apply(params, x)
    mse = jnp.mean((pred - y) ** 2)
    print(name, pred.shape, float(mse), "spike_rate", np.asarray(aux["spike_rate"]))

## Next Steps for Real VAPAR Experiments

1. Use `frame_index.csv` + split files to construct reproducible data partitions.
2. Add gaze/attention auxiliary targets if available.
3. Compare log-polar and standard conv models under the same preprocessing budget.